## Note

There are several libraries available to calculate **metrics such as Precision, Recall, and F1 Score**,  
but in this notebook, we are using **`sklearn.metrics`**.  
A working example is provided below.



In [4]:
# Step 1: Import required libraries pandas, re, sklearn.metrics
import pandas as pd
import re
from sklearn.metrics import precision_score, recall_score, f1_score

In [6]:
# Step 2: Load the CSV dataset
questions_df = pd.read_csv('Questions.csv')  # YOUR CODE HERE
questions_df.head()

,Question,Expected Answer,Actual Answer
0,What is the capital of France?,The capital of France is Paris,The capital of France is Paris.
1,Which elements were discovered by Marie Curie?,Marie Curie discovered two elements: 1. Poloni...,Marie Curie was a pioneering physicist and che...
2,By whom Pride and Prejudice was written?,Pride and Prejudice was written by Jane Austen,"Pride and Prejudice, a classic English novel, ..."
3,What is the square root of 81?,The square root of 81 is 9,The square root of 81 is 9.
4,Who won the World Cup in 1998?,Can you please tell me what\r\ntype of competi...,"France won the FIFA World Cup in 1998, which w..."


### How to Calculate Tokens

There are two main ways to calculate tokens:

1. **Manual (via OpenAI Tokenizer Tool or other calculator)**  
   - You can use the [OpenAI Tokenizer](https://platform.openai.com/tokenizer) to paste text and see exactly how it will be split into tokens for different models.  
   - This method is quick and requires no coding, but it is manual.

2. **Programmatic (via `tiktoken` (OpenAI) library or simmilar ones)**  
   - OpenAI provides the [`tiktoken`](https://github.com/openai/tiktoken) library, which allows you to calculate tokens directly in code.  
   - Example in Python:

     ```python
     import tiktoken

     # Choose encoding for your model
     encoding = tiktoken.encoding_for_model("gpt-4")

     text = "Hello world!"
     tokens = encoding.encode(text)

     print("Number of tokens:", len(tokens))
     ```

---
- You can **compare results** from the [Tokenizer UI](https://platform.openai.com/tokenizer) and the `tiktoken` library.  
- Both should give the same token count for the same model and text.  
- This is useful if you want to double-check correctness or validate your implementation.

---

### Your Task

- Calculate tokens for **expected** and **actual answers**.  
- Add two new attributes to `questions_df`:  
  - `Expected_answers_tokens`  
  - `Actual_answers_tokens`  
- We suggest using **`tiktoken`** to measure token usage and ensure consistency between the model’s input and outputs.



In [ ]:
import tiktoken

In [11]:
encoding = tiktoken.encoding_for_model("gpt-4")


def count_tokens(text):
    if pd.isna(text) or text == "":
        return 0
    return len(encoding.encode(str(text)))


questions_df['Expected_answers_tokens'] = questions_df['Expected Answer'].apply(count_tokens)
questions_df['Actual_answers_tokens'] = questions_df['Actual Answer'].apply(count_tokens)


In [12]:
questions_df

,Question,Expected Answer,Actual Answer,Expected_answers_tokens,Actual_answers_tokens
0,What is the capital of France?,The capital of France is Paris,The capital of France is Paris.,6,7
1,Which elements were discovered by Marie Curie?,Marie Curie discovered two elements: 1. Poloni...,Marie Curie was a pioneering physicist and che...,23,94
2,By whom Pride and Prejudice was written?,Pride and Prejudice was written by Jane Austen,"Pride and Prejudice, a classic English novel, ...",12,25
3,What is the square root of 81?,The square root of 81 is 9,The square root of 81 is 9.,9,10
4,Who won the World Cup in 1998?,Can you please tell me what\r\ntype of competi...,"France won the FIFA World Cup in 1998, which w...",24,36
5,Lobamba is capital of?,"Lobamba is historical capital of Eswatini, but...",I couldn't find any information on a city call...,23,45
6,Which planet is the largest in the System?,"If you meant our solar system, the largest pla...","In our solar system, Jupiter is the largest pl...",12,11


## Task: Evaluate Responses with Precision, Recall and F1 Score

Now that we can see where tokens of expected and actual answers differ,  but it just a pointing - it does not mean that outputs are incorrect.
It is useful to have detailed measurements of **Precision, Recall, and F1 Score** to evaluate responses.

We will use **`sklearn.metrics`** for this purpose.  
Here is an example showing how to calculate **Precision** for a single question:

## Example: Calculating Precision for a Single Question

We want to evaluate how well the model's answer matches the expected answer.  
Here, we use **Precision** as a metric, which measures the proportion of relevant tokens in the predicted answer.

**Question:** Who invented the telephone?  
- **Expected Answer:** Alexander Graham Bell  
- **Actual Answer:** Bell  

**Steps in the calculation:**
1. Tokenize both expected and actual answers.  
2. Create a combined list of all unique tokens.  
3. Convert each answer into a binary vector, indicating the presence (1) or absence (0) of each token.  
4. Use `sklearn.metrics.precision_score` to calculate precision.
[Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html#sklearn.metrics.precision_score) 
```python
from sklearn.metrics import precision_score 

# Step 1: Tokenization
expected_tokens = "Alexander Graham Bell".lower().split()
['alexander', 'graham', 'bell']

actual_tokens = "Bell".lower().split()
['bell']

# Step 2: Create combined list of unique tokens
all_tokens = list(set(expected_tokens).union(set(actual_tokens)))
['alexander', 'graham', 'bell']

# Step 3: Convert to binary representation for each elemt in the combined list of unique tokens and check their presence in expected_tokens and actual_tokens
# 1 if the token is present in the answer, 0 otherwise
y_true = [1 if token in expected_tokens else 0 for token in all_tokens]
[1, 1, 1] # means that all tokens from all_tokens present in expected_tokens
y_pred = [1 if token in actual_tokens else 0 for token in all_tokens]
[0, 0, 1] # means that only 1 last token from all_tokens present in actual_tokens

# Step 4: Calculate Precision
precision = precision_score(y_true, y_pred, zero_division=0)
print("Precision:", precision)
Precision: 1.0






### Your task 
is to calculate **Precision, Recall, and F1 Score** for all questions and add corresponding attributes to `questions_df`:
- `Precision`  
- `Recall`  
- `F1_Score`

After calculating tokens and evaluation metrics, the `questions_df` should have the following columns:

| Column                     | Description |
|-----------------------------|-------------|
| `Question`                  | The question text |
| `Expected Answer`           | The expected answer text |
| `Actual Answer`             | The model’s actual answer text |
| `Expected_answers_tokens`   | Number of tokens in the expected answer |
| `Actual_answers_tokens`     | Number of tokens in the actual answer |
| `Precision`                 | Precision score of the actual answer vs. expected answer |
| `Recall`                    | Recall score of the actual answer vs. expected answer |
| `F1_Score`                  | F1 Score of the actual answer vs. expected answer |

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [16]:
def calculate_tokens(expected, actual):
    # Tokenization
    expected_tokens = str(expected).lower().split()
    actual_tokens = str(actual).lower().split()

    # Combined unique tokens
    all_tokens = list(set(expected_tokens).union(set(actual_tokens)))

    # Binary vectors
    y_true = [1 if token in expected_tokens else 0 for token in all_tokens]
    y_pred = [1 if token in actual_tokens else 0 for token in all_tokens]

    return y_true, y_pred

In [26]:
y_true_res, y_pred_res = calculate_tokens(questions_df['Expected Answer'][0], questions_df['Actual Answer'][0])

print("y_true:", y_true_res)
print("y_pred:", y_pred_res)

y_true: [1, 0, 1, 1, 1, 1, 1]
y_pred: [0, 1, 1, 1, 1, 1, 1]


In [20]:
# Step 4: Calculate Precision
questions_df['Precision'] = questions_df.apply(
    lambda row: precision_score(*calculate_tokens(row['Expected Answer'], row['Actual Answer']), zero_division=0),
    axis=1
)
questions_df.head()

,Question,Expected Answer,Actual Answer,Expected_answers_tokens,Actual_answers_tokens,Precision
0,What is the capital of France?,The capital of France is Paris,The capital of France is Paris.,6,7,0.833333
1,Which elements were discovered by Marie Curie?,Marie Curie discovered two elements: 1. Poloni...,Marie Curie was a pioneering physicist and che...,23,94,0.196429
2,By whom Pride and Prejudice was written?,Pride and Prejudice was written by Jane Austen,"Pride and Prejudice, a classic English novel, ...",12,25,0.538462
3,What is the square root of 81?,The square root of 81 is 9,The square root of 81 is 9.,9,10,0.857143
4,Who won the World Cup in 1998?,Can you please tell me what\r\ntype of competi...,"France won the FIFA World Cup in 1998, which w...",24,36,0.047619


In [21]:
questions_df['Recall'] = questions_df.apply(
    lambda row: recall_score(*calculate_tokens(row['Expected Answer'], row['Actual Answer']), zero_division=0),
    axis=1
    )
questions_df.head()

,Question,Expected Answer,Actual Answer,Expected_answers_tokens,Actual_answers_tokens,Precision,Recall
0,What is the capital of France?,The capital of France is Paris,The capital of France is Paris.,6,7,0.833333,0.833333
1,Which elements were discovered by Marie Curie?,Marie Curie discovered two elements: 1. Poloni...,Marie Curie was a pioneering physicist and che...,23,94,0.196429,1.000000
2,By whom Pride and Prejudice was written?,Pride and Prejudice was written by Jane Austen,"Pride and Prejudice, a classic English novel, ...",12,25,0.538462,0.875000
3,What is the square root of 81?,The square root of 81 is 9,The square root of 81 is 9.,9,10,0.857143,0.857143
4,Who won the World Cup in 1998?,Can you please tell me what\r\ntype of competi...,"France won the FIFA World Cup in 1998, which w...",24,36,0.047619,0.058824


In [25]:
questions_df['F1_Score'] = questions_df.apply(
    lambda row: f1_score(*calculate_tokens(row['Expected Answer'], row['Actual Answer']), zero_division=0),
    axis=1
)
questions_df.head()

,Question,Expected Answer,Actual Answer,Expected_answers_tokens,Actual_answers_tokens,Precision,Recall,F1_Score
0,What is the capital of France?,The capital of France is Paris,The capital of France is Paris.,6,7,0.833333,0.833333,0.833333
1,Which elements were discovered by Marie Curie?,Marie Curie discovered two elements: 1. Poloni...,Marie Curie was a pioneering physicist and che...,23,94,0.196429,1.000000,0.328358
2,By whom Pride and Prejudice was written?,Pride and Prejudice was written by Jane Austen,"Pride and Prejudice, a classic English novel, ...",12,25,0.538462,0.875000,0.666667
3,What is the square root of 81?,The square root of 81 is 9,The square root of 81 is 9.,9,10,0.857143,0.857143,0.857143
4,Who won the World Cup in 1998?,Can you please tell me what\r\ntype of competi...,"France won the FIFA World Cup in 1998, which w...",24,36,0.047619,0.058824,0.052632
